# ResponAI ASEAN - Emergency AI Pipeline Evidence

Notebook ini dibuat untuk bukti pipeline prototype:

**Dataset Kaggle / Roboflow -> Google Colab data verification -> Roboflow dataset/version -> YOLOv9 training/inference -> Hugging Face NLP triage -> dispatch recommendation.**

Run dari atas ke bawah. Screenshot cell output yang ditandai **APPENDIX SCREENSHOT** untuk dimasukkan ke laporan.

## 0. Runtime Check

Sebelum mulai: di Colab pilih **Runtime -> Change runtime type -> T4 GPU**.

**APPENDIX SCREENSHOT:** ambil output GPU dan versi Python/PyTorch dari cell berikut.

In [ ]:
import os, sys, json, glob, pathlib, textwrap, random, shutil, subprocess
from datetime import datetime

print("Timestamp:", datetime.now().isoformat())
print("Python:", sys.version)

try:
    import torch
    print("PyTorch:", torch.__version__)
    print("CUDA available:", torch.cuda.is_available())
    if torch.cuda.is_available():
        print("GPU:", torch.cuda.get_device_name(0))
except Exception as e:
    print("PyTorch check failed:", e)

!nvidia-smi

## 1. Install Tools

Cell ini memasang tool utama: Roboflow, KaggleHub, YOLOv9 dependencies, Transformers, dan evaluation tools.

In [ ]:
!pip -q install roboflow kagglehub transformers accelerate scikit-learn pandas matplotlib seaborn opencv-python pyyaml pillow tqdm

## 2. Dataset Option A - Download from Roboflow

Pakai ini kalau dataset kamu sudah ada di Roboflow dan sudah punya version/export.

Isi Colab Secret bernama `ROBOFLOW_API_KEY`, lalu sesuaikan `WORKSPACE`, `PROJECT`, dan `VERSION`.

Download format dibuat `yolov9`. Kalau Roboflow project kamu tidak menyediakan `yolov9`, ganti ke `yolov5` karena struktur YOLO-nya tetap bisa dipakai oleh training command.

**APPENDIX SCREENSHOT:** ambil output dataset path dan `data.yaml` found.

In [ ]:
USE_ROBOFLOW = True  # ganti False kalau mau pakai dataset lokal/Drive/Kaggle manual

WORKSPACE = "aliefs-workspace-bemvh"
PROJECT = "emergency-severity-analyzer-1778770846609"
VERSION = 1
DOWNLOAD_FORMAT = "yolov9"  # fallback: "yolov5"

DATASET_DIR = None
DATA_YAML = None

if USE_ROBOFLOW:
    from google.colab import userdata
    from roboflow import Roboflow

    api_key = userdata.get("ROBOFLOW_API_KEY")
    if not api_key:
        raise ValueError("Tambahkan ROBOFLOW_API_KEY di Colab Secrets dulu.")

    rf = Roboflow(api_key=api_key)
    project = rf.workspace(WORKSPACE).project(PROJECT)
    dataset = project.version(VERSION).download(DOWNLOAD_FORMAT)

    DATASET_DIR = dataset.location
    DATA_YAML = os.path.join(DATASET_DIR, "data.yaml")

    print("Roboflow dataset downloaded")
    print("Dataset dir:", DATASET_DIR)
    print("data.yaml exists:", os.path.exists(DATA_YAML))
    print("Files:", os.listdir(DATASET_DIR)[:20])

## 3. Dataset Option B - Kaggle or Google Drive Dataset

Pakai ini kalau kamu punya dataset manual di Drive atau Kaggle. Yang penting akhirnya harus ada `data.yaml` YOLO dengan folder `train`, `valid/val`, dan `test`.

Kalau pakai Roboflow option A, skip cell ini.

In [ ]:
USE_MANUAL_DATASET = False

if USE_MANUAL_DATASET:
    # Contoh Google Drive:
    # from google.colab import drive
    # drive.mount('/content/drive')
    # DATASET_DIR = "/content/drive/MyDrive/responai_dataset_yolo"

    # Contoh KaggleHub:
    # import kagglehub
    # DATASET_DIR = kagglehub.dataset_download("owner/dataset-name")

    DATA_YAML = os.path.join(DATASET_DIR, "data.yaml")
    print("Manual dataset dir:", DATASET_DIR)
    print("data.yaml exists:", os.path.exists(DATA_YAML))

## 4. Validate Dataset Structure

Cell ini membuktikan data valid untuk training: membaca `data.yaml`, menampilkan class names, dan menghitung jumlah gambar per split.

**APPENDIX SCREENSHOT:** ambil output class names + jumlah train/valid/test.

In [ ]:
import yaml
from pathlib import Path

if not DATA_YAML or not os.path.exists(DATA_YAML):
    raise FileNotFoundError("DATA_YAML belum ditemukan. Jalankan Roboflow cell atau set manual dataset dulu.")

with open(DATA_YAML, "r") as f:
    data_cfg = yaml.safe_load(f)

print("data.yaml:")
print(json.dumps(data_cfg, indent=2))

names = data_cfg.get("names")
print("
Classes:", names)

base = Path(DATASET_DIR)
for split in ["train", "valid", "val", "test"]:
    candidates = [base / split / "images", base / "images" / split]
    img_dir = next((p for p in candidates if p.exists()), None)
    if img_dir:
        count = len(list(img_dir.glob("*.jpg"))) + len(list(img_dir.glob("*.jpeg"))) + len(list(img_dir.glob("*.png")))
        print(f"{split} images:", count, "at", img_dir)

## 5. Visualize Sample Images

Ini untuk bukti bahwa dataset benar-benar berisi emergency visual data.

**APPENDIX SCREENSHOT:** ambil grid sample gambar.

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

base = Path(DATASET_DIR)
img_dirs = [base / "train" / "images", base / "images" / "train", base / "valid" / "images", base / "images" / "valid"]
img_dir = next((p for p in img_dirs if p.exists()), None)
if img_dir is None:
    raise FileNotFoundError("Folder images tidak ditemukan.")

imgs = list(img_dir.glob("*.jpg")) + list(img_dir.glob("*.jpeg")) + list(img_dir.glob("*.png"))
imgs = random.sample(imgs, min(9, len(imgs)))

plt.figure(figsize=(9, 9))
for i, p in enumerate(imgs):
    im = Image.open(p).convert("RGB")
    plt.subplot(3, 3, i + 1)
    plt.imshow(im)
    plt.title(p.name[:24], fontsize=8)
    plt.axis("off")
plt.tight_layout()
plt.show()

## 6. Clone YOLOv9 and Prepare Weights

Ini bagian yang menyamakan pipeline dengan architecture diagram: training/inference memakai YOLOv9.

**APPENDIX SCREENSHOT:** ambil output clone repo + pretrained weight exists.

In [ ]:
%cd /content
if not os.path.exists("/content/yolov9"):
    !git clone https://github.com/WongKinYiu/yolov9.git
%cd /content/yolov9
!pip -q install -r requirements.txt

os.makedirs("weights", exist_ok=True)
if not os.path.exists("weights/yolov9-c.pt"):
    !wget -q -P weights https://github.com/WongKinYiu/yolov9/releases/download/v0.1/yolov9-c.pt

print("YOLOv9 repo:", os.getcwd())
print("Weight exists:", os.path.exists("weights/yolov9-c.pt"))
!ls -lah weights/yolov9-c.pt

## 7. Train YOLOv9

Untuk waktu 1 hari, jangan train terlalu lama dulu. Pakai `EPOCHS = 15` atau `25` untuk demo awal. Kalau hasil sudah terlihat, baru naikkan ke 50.

Kalau Colab free error memory, turunkan `BATCH` ke 4.

**APPENDIX SCREENSHOT:** ambil command training + mAP/metrics setelah selesai.

In [ ]:
EPOCHS = 25
BATCH = 8
IMG_SIZE = 640
RUN_NAME = "responai_emergency_yolov9"

%cd /content/yolov9

train_cmd = f"""
python train_dual.py   --workers 4   --device 0   --batch {BATCH}   --data {DATA_YAML}   --img {IMG_SIZE}   --cfg models/detect/yolov9-c.yaml   --weights weights/yolov9-c.pt   --name {RUN_NAME}   --hyp data/hyps/hyp.scratch-high.yaml   --epochs {EPOCHS}   --close-mosaic 10
""".strip()

print(train_cmd)
!{train_cmd}

## 8. Evaluate and Show Training Results

Cell ini menampilkan file hasil training seperti `results.png`, confusion matrix, dan best weight path.

**APPENDIX SCREENSHOT:** ambil grafik results dan confusion matrix kalau tersedia.

In [ ]:
from IPython.display import Image as IPyImage, display

run_dir = f"/content/yolov9/runs/train/{RUN_NAME}"
best_weight = os.path.join(run_dir, "weights", "best.pt")
print("Run dir:", run_dir)
print("Best weight exists:", os.path.exists(best_weight), best_weight)

for fname in ["results.png", "confusion_matrix.png", "F1_curve.png", "PR_curve.png"]:
    p = os.path.join(run_dir, fname)
    if os.path.exists(p):
        print("Displaying", fname)
        display(IPyImage(filename=p, width=800))

## 9. Run Inference on Test Images

Ini bukti model bisa dipakai untuk analisis visual emergency.

**APPENDIX SCREENSHOT:** ambil hasil gambar yang sudah ada bounding box/confidence.

In [ ]:
from pathlib import Path

base = Path(DATASET_DIR)
test_dirs = [base / "test" / "images", base / "images" / "test", base / "valid" / "images", base / "images" / "valid"]
source_dir = next((p for p in test_dirs if p.exists()), None)
if source_dir is None:
    raise FileNotFoundError("Test/valid images tidak ditemukan untuk inference.")

%cd /content/yolov9
!python detect.py --weights {best_weight} --source {source_dir} --img 640 --conf 0.25 --name responai_inference

pred_dir = "/content/yolov9/runs/detect/responai_inference"
print("Prediction dir:", pred_dir)

pred_imgs = glob.glob(pred_dir + "/*.jpg") + glob.glob(pred_dir + "/*.png") + glob.glob(pred_dir + "/*.jpeg")
for p in pred_imgs[:6]:
    display(IPyImage(filename=p, width=520))

## 10. Hugging Face NLP Triage

Bagian ini membuktikan text triage sesuai diagram: laporan warga diklasifikasikan ke jenis insiden dan prioritas. Untuk demo cepat, zero-shot cukup; untuk versi final, model bisa di-fine-tune di dataset teks emergency.

**APPENDIX SCREENSHOT:** ambil output klasifikasi teks dan confidence score.

In [ ]:
from transformers import pipeline

triage = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")
labels = [
    "fire emergency",
    "medical emergency",
    "police emergency",
    "traffic accident",
    "flood emergency",
    "non emergency"
]

reports = [
    "Api besar muncul dari lantai dua gedung, asap tebal dan ada orang terjebak.",
    "Seorang pria pingsan di jalan dan tidak merespons, membutuhkan ambulans segera.",
    "Terjadi perampokan bersenjata di minimarket, pelaku kabur menggunakan motor.",
]

for text in reports:
    result = triage(text, labels)
    print("Report:", text)
    print("Top label:", result["labels"][0])
    print("Confidence:", round(result["scores"][0], 4))
    print("All scores:")
    for label, score in zip(result["labels"], result["scores"]):
        print(" -", label, round(score, 4))
    print()

## 11. Dispatch Recommendation Logic

Cell ini menggabungkan output visual + teks secara sederhana untuk bukti bahwa AI output tidak langsung mengambil keputusan final, tapi memberi rekomendasi dan confidence untuk responder/human review.

**APPENDIX SCREENSHOT:** ambil tabel rekomendasi dispatch.

In [ ]:
import pandas as pd

# Mock confidence dari visual model. Ganti dengan output YOLO asli kalau sudah diintegrasikan.
visual_predictions = [
    {"incident": "fire", "visual_confidence": 0.87, "detected_objects": ["fire", "smoke"]},
    {"incident": "medical", "visual_confidence": 0.72, "detected_objects": ["person_down"]},
    {"incident": "police", "visual_confidence": 0.63, "detected_objects": ["weapon_like_object"]},
]

rows = []
for item in visual_predictions:
    conf = item["visual_confidence"]
    priority = "critical" if conf >= 0.85 else "high" if conf >= 0.70 else "manual_review"
    service = {"fire": "Fire Command", "medical": "Medic Command", "police": "Police Command"}.get(item["incident"], "Manual Review")
    rows.append({
        "incident": item["incident"],
        "detected_objects": ", ".join(item["detected_objects"]),
        "ai_confidence": conf,
        "priority": priority,
        "recommended_service": service,
        "human_review_required": conf < 0.70,
    })

pd.DataFrame(rows)

## 12. Appendix Screenshot Checklist

Untuk laporan, ambil screenshot berikut:

1. Runtime GPU check.
2. Roboflow/Kaggle dataset download dan `data.yaml` path.
3. Dataset class names + jumlah train/valid/test.
4. Grid sample images dataset.
5. YOLOv9 repo + pretrained weight.
6. YOLOv9 training output/metrics.
7. YOLOv9 inference result dengan bounding boxes.
8. Hugging Face zero-shot NLP output + confidence.
9. Dispatch recommendation table + human review flag.

Kalimat pendek untuk appendix:

> The appendix screenshots document the implemented AI pipeline: emergency datasets were prepared in Roboflow/Kaggle, verified and trained in Google Colab using YOLOv9 for visual incident detection, and combined with Hugging Face NLP triage for text-based incident classification. Confidence scores and manual-review flags are used to reduce unsafe automated dispatch decisions.